# Taiyi-CLIP Retrieval Threshold Evaluation

This notebook evaluates text-to-image, image-to-image, and fused text-image retrieval with Taiyi-CLIP. It computes Top-K precision and searches for the best similarity threshold by F1 score.

In [ ]:
from pathlib import Path
import json
import numpy as np

metadata_path = Path("../data/metadata/image_dataset.json")
with metadata_path.open("r", encoding="utf-8") as f:
    dataset = json.load(f)

positive_categories = ["chair", "cup", "bicycle", "skateboard", "oven"]
text_prompts = {category: f"a photo of a {category}" for category in positive_categories}
print(dataset.keys())
print(text_prompts)

## Evaluation Logic

For each positive category, the experiment compares positive image scores against negative image scores. Top-K precision is computed from ranked scores, and the best threshold is selected by F1.

In [ ]:
def binary_metrics(pos_scores, neg_scores, threshold):
    pos = np.asarray(pos_scores)
    neg = np.asarray(neg_scores)
    tp = int((pos >= threshold).sum())
    fp = int((neg >= threshold).sum())
    fn = int((pos < threshold).sum())
    precision = tp / (tp + fp) if tp + fp else 0.0
    recall = tp / (tp + fn) if tp + fn else 0.0
    f1 = 2 * precision * recall / (precision + recall) if precision + recall else 0.0
    return {"precision": precision, "recall": recall, "f1": f1}

def top_k_precision(scores_and_labels, k):
    ranked = sorted(scores_and_labels, key=lambda x: x[0], reverse=True)[:k]
    return sum(label for _, label in ranked) / k

## Model Execution

Install the optional model dependencies with `pip install -e ".[models]"`, then use `multimodal_retrieval_agent.retrieval.TaiyiClipRetriever` for actual feature extraction.